# Connect to Argilla Courses

This notebook demonstrates how to connect to the Argilla Courses instance using credentials from your `.env` file.

In [1]:
import os
import logging
from pathlib import Path
from dotenv import dotenv_values, find_dotenv
import argilla as rg
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
# Silence Argilla and Network logs globally
logging.getLogger("argilla").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

def load_credentials_from_env():
    """
    Locates the .env file using find_dotenv and loads environment variables.
    """
    env_path = find_dotenv(usecwd=True, raise_error_if_not_found=False)

    if not env_path:
        logging.error(f"Configuration file missing at: {Path.cwd() / '.env'}") 
        return {}

    env_path_obj = Path(env_path)
    
    if not env_path_obj.exists():
        logging.error(f"Configuration file missing at: {env_path_obj}")
        return {}
        
    return dotenv_values(env_path_obj)
    

ModuleNotFoundError: No module named 'argilla'

In [ ]:
env_vars = load_credentials_from_env()

argilla_courses_url = env_vars.get("ARGILLA_COURSES_API_URL")
argilla_courses_key = env_vars.get("ARGILLA_COURSES_API_KEY")

if not argilla_courses_url or not argilla_courses_key:
    logging.warning("Argilla Courses credentials incomplete. Please ensure ARGILLA_COURSES_API_URL and ARGILLA_COURSES_API_KEY are set in your .env file.")
else:
    try:
        # Initialize Client
        client = rg.Argilla(api_url=argilla_courses_url, api_key=argilla_courses_key)
        logging.info(f"Successfully connected to Argilla at {argilla_courses_url}")
        
        # Verify connection by getting current user
        print(f"Logged in as: {client.me.username}")
        
    except Exception as e:
        logging.error(f"Argilla connection failed: {e}")

In [10]:
target_workspace_name = os.getenv("WORKSPACE_TO_CREATE")
workspace = None
if target_workspace_name:
    workspace = client.workspaces(target_workspace_name)
    if not workspace:
        workspace = rg.Workspace(name=target_workspace_name)
        workspace.create()
        logging.info(f"Successfully created workspace: '{target_workspace_name}'")
    else:
        logging.debug(f"Workspace '{target_workspace_name}' already exists.")
else:
    logging.warning("'WORKSPACE_TO_CREATE' environment variable not found. Users will not be added to a workspace.")


csv_path = env_vars.get("ARGILLA_USERS_CSV_PATH")

users_df = pd.read_csv(csv_path)

# 3. Iterate through the rows and create the corresponding Argilla users
for index, row in users_df.iterrows():
    first_name = row["First Name"].strip()
    last_name = row["Last Name"].strip()
    email = row["Email Address"].strip().lower()
    
    # Set the username
    username = email.split("@")[0]
    
    # Set the password (everything before the domain string)
    password = email
    try:
        # Try to find the user first
        user = client.users(username)
        user_role = "owner"
        if user:
            user.first_name = first_name
            user.last_name = last_name
            user.role = user_role 
            user.update()
            logging.debug(f"Successfully updated user: {username} with email: {email}")
        else:
            user = rg.User(
                username=username,
                first_name=first_name,
                last_name=last_name,
                password=password,
                role=user_role
            )
            user.create()
            logging.info(f"Successfully created user: {username}")

        # Add user to the workspace if defined and created
        if workspace:
            if user not in workspace.users:
                user.add_to_workspace(workspace)
                logging.info(f"➕ Added '{username}' to workspace '{workspace.name}'")
            else:
                pass
                # logging.info(f"'{username}' is already in workspace '{workspace.name}'")


    except Exception as e:
        logging.error(f"❌ Failed to process user {username}: {e}")


In [ ]:
target_workspace_name = os.getenv("WORKSPACE_TO_CREATE")

if target_workspace_name:
    # 1. Fetch the workspace we just handled in the previous block
    workspace = client.workspaces(target_workspace_name)
    
    if workspace:
        # 2. Manage the Dataset
        dataset = client.datasets(name=target_workspace_name, workspace=workspace.name)
        
        if not dataset:            
            # Define the basic settings for your dataset
            # (Customize fields and questions based on what you actually want to annotate)
            settings = rg.Settings(
                fields=[
                    rg.TextField(name="text", title="Input Text")
                ],
                questions=[
                    rg.TextQuestion(name="notes", title="Annotator Notes", required=False)
                ]
            )
            
            dataset = rg.Dataset(
                name=target_workspace_name,
                workspace=workspace.name,
                settings=settings
            )
            dataset.create()
            logging.info(f"Successfully created dataset: '{target_workspace_name}'")
        else:
            logging.debug(f"Dataset '{target_workspace_name}' already exists.")
             
    else:
        logging.debug(f"Workspace '{target_workspace_name}' was not found. Please run the workspace creation block first.")
        
else:
    logging.debug("'WORKSPACE_TO_CREATE' environment variable not found.")
